In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
import scipy as sc
import importlib
import collision
importlib.reload(collision)
from collision import generate_spectrum, wigner_smith_matrix, wigner_smith_matrix_single, thermal_time_delay, relative_MB_weights

# parameters
e_mass = 5.48579909e-4    # electron mass in amu
t0 = 315775               # a.u. to K
tau0 = 2.418884326509e-17 # a.u. to sec

In [2]:
# define properties of molecules
reduced_masses = {"NaKNaK" : 62 / 2, "RbKRb" : 87 * (87 + 40) / (87 + 87 + 40)}     # amu
van_der_wall_coeffs = {"NaKNaK" : 561070, "RbKRb" : 8000}      # a.u.

molecules = "RbCsRbCs"
mu = 110 / e_mass      # now in a.u.
c6 = 1.9e5         # a.u.

# define natural units
beta = (2 * mu * c6) ** (1/4)    # length 
E_beta = 1 / (2 * mu * beta ** 2)   # energy
tau_beta = 2 * np.pi / E_beta    # time
print("reduced mass", mu)
print("C6", c6)
print("beta (a0)", beta)
print("E_beta (K)", E_beta * t0)
print("tau_beta (s)", tau_beta * tau0)

reduced mass 200517.73350671504
C6 190000.0
beta (a0) 525.3927746085943
E_beta (K) 2.852507331186637e-06
tau_beta (s) 1.6824646316466855e-05


In [3]:
num_samples = 5000

# properties of resonant spectrum
rrkm = 0.25e-3     # seconds
dos = rrkm / tau0 / 2 / np.pi * E_beta
mean_spacing = 1 / dos
print(mean_spacing)

# thermal properties
temp = 0.76         # in terms of E6
num_resonances = 400        # should be odd

0.06729858526586742


In [22]:
import pickle
xs = np.round(np.pow(10, np.arange(-2, 2, 0.1)), 3)

for mean_x in xs:
    mean_coupling_strength = np.sqrt(mean_x * mean_spacing / np.pi ** 2)
    time_delays = []
    closest_res = []
    for i in range(5000):
        energy_GOE_mat, W_mat, x_actual = generate_spectrum(num_resonances, mean_spacing, mean_coupling_strength)
        min_positive = np.min(energy_GOE_mat[energy_GOE_mat > 0])
        max_negative = np.max(energy_GOE_mat[energy_GOE_mat < 0])
        closest_res.append((min_positive, max_negative))

        f = lambda energy : relative_MB_weights(temp, energy) * wigner_smith_matrix_single(energy, energy_GOE_mat, np.square(W_mat))
        time_delay = sc.integrate.quad(f, 0, 10 * temp)[0]
        time_delays.append(time_delay)

    positive = [delay for delay in time_delays if delay > 0]
    frac_positive = len(positive) / len(time_delays) 

    normalized = (mean_spacing / 2 / np.pi) * np.array(positive)
    
    mean_delay = np.mean(normalized)
    std_delay = np.std(normalized)

    pkl_dict = {"delay" : time_delays, "res" : closest_res, "positive" : frac_positive, "mean" : mean_delay, "std" : std_delay, "mean_spacing" : mean_spacing, "mean_x" : mean_x}
    with open(f'RbCs/pickle/x_{mean_x}.pkl', 'wb') as f:
        pickle.dump(pkl_dict, f)

    log_delay = np.log10(normalized)
    plt.hist(log_delay, bins=30)
    plt.xlabel(f"$\\log_{{10}} (\\tau / \\tau_{{RRKM}})$")
    plt.ylabel("Counts")
    plt.title(f"x={mean_x}, Fraction={frac_positive}")
    plt.savefig(f"RbCs/figures/x_{mean_x}.pdf")
    plt.clf()



C:\Users\kevin\AppData\Local\Temp\ipykernel_51508\3950965096.py:15: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  time_delay = sc.integrate.quad(f, 0, 10 * temp)[0]
C:\Users\kevin\AppData\Local\Temp\ipykernel_51508\3950965096.py:15: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  time_delay = sc.integrate.quad(f, 0, 10 * temp)[0]
C:\Users\kevin\AppData\Local\Temp\ipykernel_51508\3950965096.py:15: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  time_delay = sc.integrate.quad(f, 0, 10 * temp)[0]
C:\Users\kevin\AppData\L

<Figure size 640x480 with 0 Axes>

In [21]:
xs = np.round(np.pow(10, np.arange(-2, 2, 0.1)), 3)
xs

array([1.0000e-02, 1.3000e-02, 1.6000e-02, 2.0000e-02, 2.5000e-02,
       3.2000e-02, 4.0000e-02, 5.0000e-02, 6.3000e-02, 7.9000e-02,
       1.0000e-01, 1.2600e-01, 1.5800e-01, 2.0000e-01, 2.5100e-01,
       3.1600e-01, 3.9800e-01, 5.0100e-01, 6.3100e-01, 7.9400e-01,
       1.0000e+00, 1.2590e+00, 1.5850e+00, 1.9950e+00, 2.5120e+00,
       3.1620e+00, 3.9810e+00, 5.0120e+00, 6.3100e+00, 7.9430e+00,
       1.0000e+01, 1.2589e+01, 1.5849e+01, 1.9953e+01, 2.5119e+01,
       3.1623e+01, 3.9811e+01, 5.0119e+01, 6.3096e+01, 7.9433e+01])